In [3]:
"""
Matrix Factorization (MF) Recommender — Explicit Ratings
=========================================================

• Training via SGD with user/item biases + L2 regularization
• Robust training loop: LR decay, LR halving on plateau, early stopping, light clipping
• Helpers: predict, top‑N recommendation, similar items, RMSE evaluation
• Big synthetic dataset generator to stress‑test model behavior

USAGE
-----
$ python mf_recommender.py

REQUIREMENTS
------------
- numpy
- pandas
- scikit-learn (only for train_test_split in the demo)

TIPS
----
- If RMSE increases early, lower lr, raise reg slightly, or increase patience.
- For very large data, consider mini-batching or switching to an optimizer like Adam.
- For *implicit* feedback (clicks/views), consider Weighted-ALS (Hu–Koren–Volinsky) instead of this explicit-RMSE objective.
"""

from __future__ import annotations
from dataclasses import dataclass
from typing import Optional, Tuple, List, Dict

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split


# =========================
#  Configuration Dataclass
# =========================
@dataclass
class MFConfig:
    """Container for hyperparameters and training knobs.

    Attributes
    ----------
    n_factors : int
        Latent dimensionality (rank) of user/item embeddings (P/Q). Higher → more capacity.
    n_epochs : int
        Max epochs (one pass over all observed ratings, shuffled).
    lr : float
        Initial learning rate for SGD updates.
    reg : float
        L2 regularization coefficient applied to P, Q, and biases (bu/bi).
    init_std : float
        Stddev for Gaussian initialization of P and Q.
    early_stopping : bool
        Stop when validation RMSE does not improve for `es_patience` epochs.
    es_patience : int
        Allowed epochs without validation improvement before stopping.
    seed : int
        Random seed to make results reproducible (affects init and shuffling).
    min_rating / max_rating : Optional[float]
        Optional clipping bounds for predictions (e.g., 1..5 for star ratings).
    """

    n_factors: int = 64
    n_epochs: int = 40
    lr: float = 0.01
    reg: float = 0.05
    init_std: float = 0.05
    early_stopping: bool = True
    es_patience: int = 5
    seed: int = 42
    min_rating: Optional[float] = 1.0
    max_rating: Optional[float] = 5.0


# ==================================
#  Matrix Factorization Recommender
# ==================================
class MatrixFactorizationRecommender:
    """User–Item Matrix Factorization with biases.

    Predicts:
        r_hat(u, i) = global_mean + bu[u] + bi[i] + <P[u], Q[i]>

    Notes
    -----
    - Unknown users/items at inference time back off to global mean + whichever bias(es) are known.
    - Targets *explicit* ratings with squared-error (RMSE) objective.
    - For production cold-start mitigation, combine with metadata models or default/popularity priors.
    """

    # ---------- Initialization ----------
    def __init__(self, config: MFConfig = MFConfig()):
        self.cfg = config

        # External ID ↔ internal index mappings (dense arrays are faster than dicts for math)
        self.user_map: Dict = {}
        self.item_map: Dict = {}
        self.inv_user_map: Dict = {}
        self.inv_item_map: Dict = {}

        # Learnable parameters
        self.global_mean: float = 0.0
        self.P: Optional[np.ndarray] = None  # (n_users, k)
        self.Q: Optional[np.ndarray] = None  # (n_items, k)
        self.bu: Optional[np.ndarray] = None # (n_users,)
        self.bi: Optional[np.ndarray] = None # (n_items,)

        self._is_fit = False
        np.random.seed(self.cfg.seed)

    # ---------- Public API ----------
    def fit(
        self,
        df: pd.DataFrame,
        user_col: str,
        item_col: str,
        rating_col: str,
        val_df: Optional[pd.DataFrame] = None,
    ) -> "MatrixFactorizationRecommender":
        """Fit on a DataFrame with columns [user_col, item_col, rating_col].

        If `val_df` is None, a 90/10 split is created from `df` for early stopping.
        """
        assert set([user_col, item_col, rating_col]).issubset(df.columns)

        # Map original user/item ids to contiguous integers
        users = pd.Categorical(df[user_col])
        items = pd.Categorical(df[item_col])

        self.user_map = {u: i for i, u in enumerate(users.categories)}
        self.item_map = {i: j for j, i in enumerate(items.categories)}
        self.inv_user_map = {i: u for u, i in self.user_map.items()}
        self.inv_item_map = {j: i for i, j in self.item_map.items()}

        # Convert to index + numpy arrays for speed
        u_idx = users.codes.astype(np.int64)
        i_idx = items.codes.astype(np.int64)
        r = df[rating_col].astype(np.float32).values

        n_users = len(self.user_map)
        n_items = len(self.item_map)
        k = self.cfg.n_factors

        # Initialize P, Q, and biases
        self.P = np.random.normal(0, self.cfg.init_std, size=(n_users, k)).astype(np.float32)
        self.Q = np.random.normal(0, self.cfg.init_std, size=(n_items, k)).astype(np.float32)
        self.bu = np.zeros(n_users, dtype=np.float32)
        self.bi = np.zeros(n_items, dtype=np.float32)
        self.global_mean = float(np.mean(r))

        # Train/Validation split
        if val_df is None:
            train_idx, val_idx = train_test_split(
                np.arange(len(df)), test_size=0.1, random_state=self.cfg.seed
            )
            train_data = (u_idx[train_idx], i_idx[train_idx], r[train_idx])
            val_data   = (u_idx[val_idx],   i_idx[val_idx],   r[val_idx])
        else:
            # Map validation ids via the train mappings; drop pairs unseen in train.
            val_u = val_df[user_col].map(self.user_map).values
            val_i = val_df[item_col].map(self.item_map).values
            mask = (~pd.isna(val_u)) & (~pd.isna(val_i))
            val_u = val_u[mask].astype(int)
            val_i = val_i[mask].astype(int)
            val_r = val_df.loc[mask, rating_col].astype(np.float32).values
            train_data = (u_idx, i_idx, r)
            val_data   = (val_u, val_i, val_r)

        self._sgd_train(train_data, val_data)
        self._is_fit = True
        return self

    def predict(self, user_id, item_id) -> float:
        """Predict rating for a (user_id, item_id).

        Unknown user or item falls back to the sum of whatever components are known
        (global mean, user bias, item bias). The latent interaction term requires both ids known.
        """
        self._check_fit()
        u = self.user_map.get(user_id, None)
        i = self.item_map.get(item_id, None)

        pred = self.global_mean
        if u is not None:
            pred += self.bu[u]
        if i is not None:
            pred += self.bi[i]
        if (u is not None) and (i is not None):
            pred += float(self.P[u] @ self.Q[i])
        return self._clip(pred)

    def recommend_for_user(
        self,
        user_id,
        N: int = 10,
        exclude_seen: bool = True,
        seen_items: Optional[pd.Series] = None,
    ) -> List[Tuple[object, float]]:
        """Return top‑N item recommendations for a given user.

        Overview
        --------
        For a user `u`, compute a score for *every* item `i` using the model:

            score(u, i) = global_mean + bu[u] + bi[i] + <P[u], Q[i]>

        where
        - global_mean is the average rating across all training data
        - bu[u] is the learned bias for user u (e.g., strict/lenient rater)
        - bi[i] is the learned bias for item i (e.g., generally loved/hated item)
        - <P[u], Q[i]> is the latent interaction between user & item

        After computing all scores, we select the top‑N highest‑scoring items.

        Cold‑Start (Unknown User)
        -------------------------
        If `user_id` wasn't in training, P[u] and bu[u] are unavailable.
        We fall back to ranking items by:  global_mean + bi[i].

        Excluding Already‑Rated Items
        -----------------------------
        If `exclude_seen=True` and you provide the user's previously seen items
        via `seen_items` (Series of item ids), those indices are masked (score = -∞)
        so recommendations favor *new* items.
        """
        self._check_fit()
        u = self.user_map.get(user_id, None)

        # Case 1: Unknown user → cold‑start fallback: rank by (global mean + item bias)
        if u is None:
            scores = self.global_mean + self.bi
            # argsort returns ascending indices; reverse to get highest scores first
            top_i = np.argsort(scores)[::-1][:N]
            return [(self.inv_item_map[i], float(scores[i])) for i in top_i]

        # Case 2: Known user → compute scores for *all* items efficiently
        # P[u] is (k,), Q is (n_items, k).  P[u] @ Q.T gives dot products with all item vectors at once.
        
        scores = self.global_mean + self.bu[u] + self.bi + (self.P[u] @ self.Q.T)

        # Optionally mask items that user already consumed/rated
        if exclude_seen and seen_items is not None and len(seen_items) > 0:
            # Map external item IDs → internal indices, drop unknowns, then mask
            seen_idx = seen_items.map(self.item_map).dropna().astype(int).values
            scores[seen_idx] = -np.inf

        # Efficient Top‑N selection:
        # np.argpartition finds the N largest elements without fully sorting the entire array
        # (substantially faster when there are many items). Then sort just those N.
        top_i = np.argpartition(-scores, kth=min(N, len(scores) - 1))[:N]
        top_i = top_i[np.argsort(-scores[top_i])]

        # Convert internal indices back to original item IDs and clip to rating bounds if configured
        return [(self.inv_item_map[i], float(self._clip(scores[i]))) for i in top_i]

    def similar_items(self, item_id, N: int = 10) -> List[Tuple[object, float]]:
        """Return top‑N most similar items by cosine similarity in item‑factor space (Q).

        Cosine similarity is scale‑invariant; good for identifying neighborhoods.
        """
        self._check_fit()
        i = self.item_map.get(item_id, None)
        if i is None:
            return []
        qi = self.Q[i]
        norms = np.linalg.norm(self.Q, axis=1) * (np.linalg.norm(qi) + 1e-12)
        sims = (self.Q @ qi) / (norms + 1e-12)
        sims[i] = -np.inf  # exclude self
        top = np.argpartition(-sims, kth=min(N, len(sims) - 1))[:N]
        top = top[np.argsort(-sims[top])]
        return [(self.inv_item_map[j], float(sims[j])) for j in top]

    def evaluate_rmse(self, df: pd.DataFrame, user_col: str, item_col: str, rating_col: str) -> float:
        """Compute RMSE on a DataFrame. Unknown users/items are handled gracefully."""
        preds = [self.predict(u, i) for u, i in zip(df[user_col], df[item_col])]
        true = df[rating_col].values.astype(np.float32)
        return float(np.sqrt(np.mean((np.array(preds, dtype=np.float32) - true) ** 2)))

    # ---------- Internal: Robust SGD Training Loop ----------
    def _sgd_train(self, train_data, val_data):
        """Stochastic Gradient Descent over observed (u, i, r) triples.

        Includes:
        - Per‑epoch LR decay
        - Auto LR halving when validation RMSE plateaus
        - Early stopping restoring best parameters
        - Light gradient clipping on factors and biases for stability
        """
        u_idx, i_idx, r = train_data
        val_u, val_i, val_r = val_data
        n = len(r)

        # Base hyperparams
        lr0 = float(self.cfg.lr)  # initial learning rate
        lr = lr0
        reg = float(self.cfg.reg)

        best_rmse = np.inf
        best_state = None
        no_improve_epochs = 0
        plateau_epochs = 0

        # Light clipping thresholds — enough to prevent blowups without constraining learning too much
        clip_vec = 5.0   # clip each latent vector element into [-5, 5]
        clip_bias = 5.0  # clip biases into [-5, 5]

        for epoch in range(1, self.cfg.n_epochs + 1):
            # Shuffle training triples each epoch to make updates i.i.d.-ish
            perm = np.random.permutation(n)

            for t in perm:
                u, i, y = u_idx[t], i_idx[t], r[t]

                # Current prediction and error for this pair
                pred = self.global_mean + self.bu[u] + self.bi[i] + self.P[u].dot(self.Q[i])
                err = y - pred  # gradient direction is proportional to +err (we subtract grad)

                # Cache copies before in-place updates (needed for cross terms)
                Pu = self.P[u].copy()
                Qi = self.Q[i].copy()

                # Bias updates (L2 regularized)
                self.bu[u] += lr * (err - reg * self.bu[u])
                self.bi[i] += lr * (err - reg * self.bi[i])

                # Latent factor updates (L2 regularized)
                # d/dPu = -err * Qi + reg * Pu  ->  Pu += lr * (err * Qi - reg * Pu)
                # d/dQi = -err * Pu + reg * Qi  ->  Qi += lr * (err * Pu - reg * Qi)
                self.P[u]  += lr * (err * Qi - reg * Pu)
                self.Q[i]  += lr * (err * Pu - reg * Qi)

                # Clip to keep parameters within a safe range (prevents rare runaway steps)
                if clip_vec is not None:
                    np.clip(self.P[u], -clip_vec, clip_vec, out=self.P[u])
                    np.clip(self.Q[i], -clip_vec, clip_vec, out=self.Q[i])
                if clip_bias is not None:
                    if self.bu[u] > clip_bias:
                        self.bu[u] = clip_bias
                    elif self.bu[u] < -clip_bias:
                        self.bu[u] = -clip_bias
                    if self.bi[i] > clip_bias:
                        self.bi[i] = clip_bias
                    elif self.bi[i] < -clip_bias:
                        self.bi[i] = -clip_bias

            # ---- End of epoch: diagnostics ----
            train_preds = (
                self.global_mean + self.bu[u_idx] + self.bi[i_idx] + np.sum(self.P[u_idx] * self.Q[i_idx], axis=1)
            )
            train_rmse = float(np.sqrt(np.mean((r - train_preds) ** 2)))

            val_preds = (
                self.global_mean + self.bu[val_u] + self.bi[val_i] + np.sum(self.P[val_u] * self.Q[val_i], axis=1)
            )
            val_rmse = float(np.sqrt(np.mean((val_r - val_preds) ** 2)))

            print(f"Epoch {epoch:02d} | lr {lr:.5f} | train RMSE {train_rmse:.4f} | val RMSE {val_rmse:.4f}")

            # Keep best snapshot
            if val_rmse + 1e-6 < best_rmse:
                best_rmse = val_rmse
                best_state = (self.P.copy(), self.Q.copy(), self.bu.copy(), self.bi.copy())
                no_improve_epochs = 0
                plateau_epochs = 0
            else:
                no_improve_epochs += 1
                plateau_epochs += 1

            # If we plateau for 2 consecutive epochs, cut LR in half (helps when initial lr is too large)
            if plateau_epochs >= 2:
                new_lr = max(lr * 0.5, lr0 * 1e-3)  # don't decay below 0.001 * lr0
                if new_lr < lr:
                    lr = new_lr
                plateau_epochs = 0

            # Mild per-epoch decay (independent of plateau logic)
            lr = max(lr * 0.98, lr0 * 1e-3)

            # Early stopping
            if self.cfg.early_stopping and no_improve_epochs >= self.cfg.es_patience:
                print(f"Early stopping at epoch {epoch}. Best val RMSE = {best_rmse:.4f}")
                break

        # Restore best parameters to avoid ending on a worse snapshot
        if best_state is not None:
            self.P, self.Q, self.bu, self.bi = best_state

    # ----- Utilities -----
    def _clip(self, x: float) -> float:
        """Apply optional clipping of predictions to rating bounds."""
        if self.cfg.min_rating is not None:
            x = max(x, self.cfg.min_rating)
        if self.cfg.max_rating is not None:
            x = min(x, self.cfg.max_rating)
        return x

    def _check_fit(self):
        if not self._is_fit:
            raise RuntimeError("Model not fitted yet. Call fit() first.")


# ==================================
#  Synthetic Data (Explicit Ratings)
# ==================================

def generate_synthetic_ratings(
    n_users: int = 2000,
    n_items: int = 1500,
    n_factors_true: int = 32,
    density: float = 0.004,         # fraction of observed entries (~0.4% default)
    noise_std: float = 0.30,
    seed: int = 2025,
    rating_min: float = 1.0,
    rating_max: float = 5.0,
) -> pd.DataFrame:
    """Create a synthetic explicit-rating dataset with known ground-truth factors.

    Parameters
    ----------
    n_users, n_items : int
        Problem size. Overall rating matrix is (n_users x n_items).
    n_factors_true : int
        Latent dimensionality used to *generate* the data (can differ from model rank).
    density : float
        Fraction of (user, item) pairs that will be observed (sparsity control).
    noise_std : float
        Standard deviation of Gaussian noise added to the ground-truth ratings.
    seed : int
        RNG seed for reproducibility.
    rating_min, rating_max : float
        Min/max clipping for ratings (e.g., 1..5 stars).

    Returns
    -------
    DataFrame with columns: ["user", "item", "rating"].
    """
    rng = np.random.default_rng(seed)

    # Ground-truth latent structure and biases
    U_true = rng.normal(0, 1.0, size=(n_users, n_factors_true))
    V_true = rng.normal(0, 1.0, size=(n_items, n_factors_true))
    bu_true = rng.normal(0, 0.3, size=n_users)
    bi_true = rng.normal(0, 0.3, size=n_items)
    mu_true = 3.5  # realistic global mean near middle of 1..5

    # Number of observed pairs
    total_pairs = n_users * n_items
    m = int(total_pairs * density)

    # Sample unique pairs uniformly over the matrix (without replacement)
    choices = rng.choice(total_pairs, size=m, replace=False)
    u_idx = choices // n_items
    i_idx = choices %  n_items

    # Generate ratings = mu + bu + bi + <U, V> + noise, then clip
    dot = np.sum(U_true[u_idx] * V_true[i_idx], axis=1)
    y = mu_true + bu_true[u_idx] + bi_true[i_idx] + dot + rng.normal(0, noise_std, size=m)
    y = np.clip(y, rating_min, rating_max)

    users = [f"u{u}" for u in u_idx]
    items = [f"i{i}" for i in i_idx]
    df = pd.DataFrame({"user": users, "item": items, "rating": y.astype(np.float32)})
    return df


# ================
#  Demo Entrypoint
# ================
if __name__ == "__main__":
    print("Generating synthetic dataset…")
    df = generate_synthetic_ratings()
    print(df.head())
    print("Dataset size:", len(df), "ratings")

    # Train/validation split (explicitly pass val_df to fit)
    train_df, val_df = train_test_split(df, test_size=0.1, random_state=0)

    # Model hyperparameters — feel free to tweak
    cfg = MFConfig(
        n_factors=64,
        n_epochs=50,
        lr=0.02,      # try 0.01 if you see early divergence
        reg=0.06,     # increase to 0.08–0.12 if overfitting
        early_stopping=True,
        es_patience=5,
        min_rating=1.0,
        max_rating=5.0,
        seed=123,
    )

    print("Training MF model…")
    mf = MatrixFactorizationRecommender(cfg).fit(
        train_df, user_col="user", item_col="item", rating_col="rating", val_df=val_df
    )

    # Evaluation on validation set
    rmse = mf.evaluate_rmse(val_df, user_col="user", item_col="item", rating_col="rating")
    print(f"Validation RMSE: {rmse:.4f}")

    # Sample predictions and recommendations
    sample_user = "u0"
    sample_item = "i0"
    print(f"Pred({sample_user}, {sample_item}) =", mf.predict(sample_user, sample_item))

    seen_items_u0 = train_df.loc[train_df["user"] == sample_user, "item"]
    recs = mf.recommend_for_user(sample_user, N=10, exclude_seen=True, seen_items=seen_items_u0)
    print(f"Top-10 for {sample_user}:", recs)

    sims = mf.similar_items("i0", N=10)
    print("Items similar to i0:", sims)


Generating synthetic dataset…
    user   item    rating
0  u1860  i1196  4.377960
1   u238  i1322  1.458967
2  u1822   i827  1.000000
3     u5   i475  5.000000
4  u1545   i154  3.670677
Dataset size: 12000 ratings
Training MF model…
Epoch 01 | lr 0.02000 | train RMSE 1.7249 | val RMSE 1.7889
Epoch 02 | lr 0.01960 | train RMSE 1.6676 | val RMSE 1.8011
Epoch 03 | lr 0.01921 | train RMSE 1.6220 | val RMSE 1.8150
Epoch 04 | lr 0.00941 | train RMSE 1.6025 | val RMSE 1.8221
Epoch 05 | lr 0.00922 | train RMSE 1.5846 | val RMSE 1.8293
Epoch 06 | lr 0.00452 | train RMSE 1.5762 | val RMSE 1.8327
Early stopping at epoch 6. Best val RMSE = 1.7889
Validation RMSE: 1.7867
Pred(u0, i0) = 3.1025214
Top-10 for u0: [('i996', 3.4138290882110596), ('i282', 3.3902993202209473), ('i891', 3.3786540031433105), ('i56', 3.3593032360076904), ('i1309', 3.352928400039673), ('i547', 3.3464696407318115), ('i35', 3.3453452587127686), ('i629', 3.3441505432128906), ('i1311', 3.3426191806793213), ('i695', 3.340317249298